# YOLO 객체 탐지 실습
[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimhaneul050126-cmyk/codeplant-ai-vision-lab/blob/main/02_YOLO_객체탐지_실습.ipynb)
MNIST에서는 사진 전체에 대해 숫자 하나를 예측했습니다. 여기서는 사진 안의 물체마다 **종류, 위치, 탐지 점수**를 확인합니다.

위에서부터 셀을 차례로 실행하세요. **기본 CPU 런타임**으로 충분합니다(GPU 변경 불필요). 첫 실행에는 패키지 설치와 모델 다운로드로 1~2분 걸립니다.

In [ ]:
# 수업용으로 동작을 확인한 버전으로 고정합니다
%pip -q install ultralytics==8.4.207

## 1. 모델 불러오기
공개 이미지 데이터(COCO, 80가지 물체)로 미리 학습된 **YOLO26n**을 사용합니다. 예제 사진은 패키지 안에 들어 있어 인터넷 연결 상태와 상관없이 열립니다.

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps
from ultralytics import YOLO
from ultralytics.utils import ASSETS

model = YOLO("yolo26n.pt")
bus_photo = str(ASSETS / "bus.jpg")        # 버스와 사람들
people_photo = str(ASSETS / "zidane.jpg")  # 두 사람

def detect(photo, threshold=0.25):
    """사진 한 장을 탐지하고, 점수가 threshold 이상인 박스만 남긴 결과를 돌려줍니다."""
    return model.predict(source=photo, conf=threshold, device="cpu", verbose=False)[0]

def show(result):
    """박스가 그려진 사진을 화면에 표시합니다."""
    display(result.plot(pil=True))

def to_table(result):
    """탐지 결과를 표(종류·점수·좌표)로 바꿉니다."""
    rows = []
    for box in result.boxes.cpu():
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        rows.append({
            "종류": result.names[int(box.cls.item())],
            "탐지 점수": round(box.conf.item(), 3),
            "왼쪽 x": round(x1), "위 y": round(y1),
            "오른쪽 x": round(x2), "아래 y": round(y2),
        })
    columns = ["종류", "탐지 점수", "왼쪽 x", "위 y", "오른쪽 x", "아래 y"]
    return pd.DataFrame(rows, columns=columns)

print("모델이 아는 물체 수:", len(model.names))

## 2. 예제 사진에서 박스와 점수 읽기
박스는 모델이 예측한 위치입니다. 박스 위의 글자는 물체 종류이고, 숫자는 **탐지 점수**입니다.

- 탐지 점수는 "정답일 확률"이 아니라 **모델이 얼마나 확신하는지**를 나타내는 값입니다.
- 몇 점 이상을 결과로 인정할지(**기준, conf**)는 우리가 정합니다.
- 표의 좌표는 이미지 왼쪽 위를 (0, 0)으로 한 픽셀 위치입니다.

In [ ]:
result = detect(bus_photo)
show(result)
display(to_table(result))

## 3. 같은 사진, 다른 점수 기준
기준을 바꾸면 남는 박스가 달라집니다.

- 기준을 **낮추면**: 놓친 물체를 더 찾지만, 엉뚱한 박스(**오검출**)도 늘어납니다.
- 기준을 **높이면**: 엉뚱한 박스는 줄지만, 실제 물체를 놓치는 경우(**미검출**)가 늘어납니다.

세 그림을 비교해 **어떤 박스가 언제 사라지는지** 물체 이름과 점수를 함께 기록해 보세요.

In [ ]:
thresholds = (0.10, 0.25, 0.70)

fig, axes = plt.subplots(1, len(thresholds), figsize=(15, 7))
for ax, threshold in zip(axes, thresholds):
    prediction = detect(bus_photo, threshold)
    ax.imshow(prediction.plot(pil=True))
    ax.set_title(f"conf = {threshold:.2f} | {len(prediction.boxes)} boxes")
    ax.axis("off")
plt.tight_layout()
plt.show()

# 가장 낮은 기준에서 잡힌 박스 전체를 점수 순으로 확인
display(to_table(detect(bus_photo, thresholds[0])).sort_values("탐지 점수", ascending=False))

## 4. 다른 사진과 비교하기
같은 기준으로 두 번째 사진을 실행합니다. 사진 속 물체의 크기나 가려진 정도가 달라지면 탐지 결과도 달라질 수 있습니다.

In [ ]:
second_result = detect(people_photo, 0.25)
show(second_result)
display(to_table(second_result))

## 5. 탐지 결과를 RC카 명령으로 바꾸기
탐지 결과만으로는 차가 움직이지 않습니다. 결과를 보고 **명령(w 전진 / x 정지)**을 고르는 판단 코드가 필요합니다.

아래 함수는 사람이나 정지 표지판이 기준 점수 이상으로 보이면 정지(`x`), 아니면 전진(`w`)을 고릅니다. `stop_classes`와 `threshold`를 바꿔 보며 명령이 어떻게 달라지는지 확인하세요.

In [ ]:
def decide_from_detection(result, stop_classes=("person", "stop sign"), threshold=0.5):
    """멈춰야 할 물체가 기준 점수 이상으로 보이면 'x'(정지), 아니면 'w'(전진)."""
    for box in result.boxes:
        name = result.names[int(box.cls.item())]
        score = box.conf.item()
        if name in stop_classes and score >= threshold:
            print(f"정지 이유: {name} (점수 {score:.2f})")
            return "x"
    return "w"

# 낮은 기준(0.10)으로 탐지한 결과를 판단 함수에 넘깁니다
low_result = detect(bus_photo, 0.10)
print("명령:", decide_from_detection(low_result))
print("명령 (정지 표지판만, 기준 0.1):", decide_from_detection(low_result, stop_classes=("stop sign",), threshold=0.1))
print("명령 (정지 표지판만, 기준 0.5):", decide_from_detection(low_result, stop_classes=("stop sign",), threshold=0.5))

## 6. 내 사진으로 실행하기 (선택)
JPG 또는 PNG 한 장을 올립니다. 모델이 놓친 물체가 있다면 `threshold`를 낮추고, 어떤 박스가 새로 표시되는지 비교해 보세요.

In [ ]:
from google.colab import files

threshold = 0.25
uploaded = files.upload()
if len(uploaded) == 1:
    photo_bytes = next(iter(uploaded.values()))
    with Image.open(BytesIO(photo_bytes)) as image:
        photo = ImageOps.exif_transpose(image).convert("RGB")
    custom_result = detect(photo, threshold)
    show(custom_result)
    display(to_table(custom_result))
    print("명령:", decide_from_detection(custom_result))
else:
    print("이미지 한 장을 선택해 셀을 다시 실행하세요.")

## 결과 정리와 RC카 연결
- 기준을 올리면 어떤 박스가 사라졌는지 **물체 이름과 점수**를 함께 기록합니다.
- **미검출**: 실제로 있는 물체를 찾지 못한 경우입니다. **오검출**: 없는 물체를 있다고 한 경우입니다. 기준값은 둘 사이의 균형을 고르는 일입니다.
- 이 모델은 신호등(`traffic light`)의 **위치**는 찾지만 빨간불·초록불은 구분하지 않습니다. 색 판별은 별도로 하고, 그 결과를 **전진·정지 판단 코드**에 넘깁니다.
- 안전이 중요한 RC카에서는 "확신이 낮으면 멈춘다"가 기본입니다. 정지 판단의 기준은 낮게, 전진 판단의 기준은 높게 두는 이유를 생각해 보세요.

참고 자료 · [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26) · [Ultralytics Predict 모드](https://docs.ultralytics.com/modes/predict)

## 확장 실습 · 화면 신호등
회로 없이 Colab 화면에서 **AI 판단 → 출력**을 확인합니다. 판단 함수가 고른 글자를 `show_light`에 넣으면 신호등이 그려집니다.

| 명령 | 뜻 | 신호등 |
|---|---|---|
| `w` | 전진 | 초록 |
| `x` | 정지 | 빨강 |
| 그 밖의 글자 | 약속하지 않은 입력 | **빨강 (안전하게 정지)** |

In [ ]:
from IPython.display import HTML, display

def show_light(cmd):
    """명령 글자 하나를 받아 화면에 신호등을 그립니다. 약속하지 않은 입력은 정지로 처리합니다."""
    go = (cmd == "w")
    label = "w · 전진" if go else ("x · 정지" if cmd == "x" else f"{cmd!r} · 모르는 명령 → 정지")
    red = "#e53935" if not go else "#3a3a3a"
    green = "#43a047" if go else "#3a3a3a"
    display(HTML(f'''
    <div style="display:flex;align-items:center;gap:24px;font-family:sans-serif">
      <div style="background:#222;border-radius:18px;padding:14px;display:flex;flex-direction:column;gap:12px">
        <div style="width:64px;height:64px;border-radius:50%;background:{red};box-shadow:0 0 {24 if not go else 0}px {red}"></div>
        <div style="width:64px;height:64px;border-radius:50%;background:{green};box-shadow:0 0 {24 if go else 0}px {green}"></div>
      </div>
      <div style="font-size:28px;font-weight:700">{label}</div>
    </div>'''))
    return "w" if go else "x"

print("show_light 준비 완료")

### 단계 1 · 신호등 표시 테스트
명령 글자를 직접 넣어 봅니다. `'q'`처럼 약속하지 않은 글자는 왜 정지로 처리하는 것이 안전할까요?

In [ ]:
show_light("w")
show_light("x")
show_light("q")

### 단계 2 · AI 판단 결과를 신호등으로
위의 `decide_from_detection()` 결과를 그대로 넣습니다. 기준 점수와 멈출 물체를 바꿔 신호등이 어떻게 달라지는지 비교하세요.

In [ ]:
low_result = detect(bus_photo, 0.10)

print("① 사람·정지 표지판 기준 0.5")
show_light(decide_from_detection(low_result))

print("② 정지 표지판만, 기준 0.1")
show_light(decide_from_detection(low_result, stop_classes=("stop sign",), threshold=0.1))

print("③ 정지 표지판만, 기준 0.5")
show_light(decide_from_detection(low_result, stop_classes=("stop sign",), threshold=0.5))

### 관찰과 RC카 연결
- 같은 사진인데도 **멈출 물체와 기준 점수**에 따라 신호등이 바뀝니다.
- 실제 RC카 조종기도 시리얼로 글자 하나(`w`/`x`)를 받아 무선으로 보내고, 0.7초 동안 새 명령이 없으면 정지로 돌아갑니다.
- 화면 신호등 대신 이 글자를 조종기로 보내면 RC카가 움직입니다.